In [ ]:
#import libraries


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.ensemble import RandomForestRegressor


import os
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

print("Path to dataset files:", path)


csv_path = os.path.join(path, "Q1_data.csv")

og_df = pd.read_csv(csv_path)



In [ ]:
df = og_df.copy()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()




In [ ]:
target_column = 'Delivery_Time'
check_target_distribution(df, target_column)

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

# Droppin rows with missing target or missing important feature
df = df.dropna(subset=['Delivery_Time', 'Traffic_Level'])

# filling with mean because no outliers
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

#filling with mode for categorical
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])


In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Encode Categorical Features
encoder = LabelEncoder()

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = encoder.fit_transform(df[col])



In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()

# We want to scale all the columns, BUT NOT THE TARGET
cols = df.columns.drop("Delivery_Time")

df[cols] = scaler.fit_transform(df[cols])

In [ ]:
# Task 6: Write your code here:
# no need for balance checking because this is a regression

In [ ]:
# Task 1: Write your code here:
target_column = "Delivery_Time"

X = df.drop(target_column, axis=1)
y = df[target_column]

print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:


model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

# Training on KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)


mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx,:], X.iloc[test_idx,:]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation \n" + "-"*40)
print(f"Average MAE across all folds : {np.mean(mae_scores):.2f}")
print("-"*40)

In [ ]:
# Compare model to baseline
baseline_pred = np.full_like(y, y.median())

baseline_mae = mean_absolute_error(y, baseline_pred)


print(f"Baseline MAE (using median target): {baseline_mae:.4f}")




In [ ]:
# Task 1: Write your code here:

importance = list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance, key=lambda x: abs(x[1]), reverse=True)

# Extract sorted features and their coefficients
features, coefficients = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Regression Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:

y_pred = model.predict(X)
prediction_df = pd.DataFrame(y_pred)
prediction_df.columns = ['Predicted Delivery Time']


check_target_distribution(prediction_df, 'Predicted Delivery Time')

In [ ]:
!pip install dask[dataframe] catboost


In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor



model1 = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model2 = CatBoostRegressor(verbose=0)
# Training on KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)


mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx,:], X.iloc[test_idx,:]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model1.fit(X_train, y_train)
    model2.fit(X_train, y_train)

    # Predict
    y_pred1 = model1.predict(X_test)
    y_pred2 = model2.predict(X_test)
    y_pred = (y_pred1 + y_pred2) / 2


    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation \n" + "-"*40)
print(f"Average MAE across all folds : {np.mean(mae_scores):.2f}")
print("-"*40)
